### 라이브러리 선언 및 설치 

In [1]:
#라이브러리 설치
!pip install pandas
!pip install sqlalchemy
!python -m pip install --trusted-host pypi.python.org --trusted-host pypi.org --trusted-host files.pythonhosted.org pymysql
!pip install -U nest-asyncio==1.6.0 pyngrok==7.2.4 uvicorn==0.34.2 fastapi==0.115.12

In [2]:
#라이브러리 선언

#데이터 처리 라이브러리
import pandas as pd

#Db 연동 라이브러리
from sqlalchemy import create_engine 

#서버 관리용 fastapi 의존 라이브러리
import uvicorn
import nest_asyncio

#fast api 라이브러리
from fastapi import FastAPI 

#인터페이스 데이트 관리 라이브러리
from pydantic import BaseModel

#cors 라이브러리
from fastapi.middleware.cors import CORSMiddleware

### 데이터 파일 확인

In [3]:
# csv 파일 읽기
df = pd.read_csv('소상공인시장진흥공단_상가(상권)정보_서울_202512.csv', encoding='cp949') 
#데이터 상위 확인 
print(df.head(1)) 

       상호명  지점명 상권업종대분류코드 상권업종대분류명 상권업종중분류코드 상권업종중분류명 상권업종소분류코드 상권업종소분류명  \
0  60계치킨암사  선사점        I2       음식      I210    기타 간이    I21006       치킨   

  표준산업분류코드 표준산업분류명  ...              도로명 건물본번지  건물부번지      건물명  \
0   I56193  치킨 전문점  ...  서울특별시 강동구 상암로3길   8.0    NaN  암사동정웅빌딩   

               도로명주소   구우편번호  신우편번호 동정보  층정보  호정보  
0  서울특별시 강동구 상암로3길 8  134877   5241 NaN    1  NaN  

[1 rows x 31 columns]


In [4]:
#상권 리스트 확인 (unique 중복제거)
regions = df['상권업종대분류명'].unique()
regions

<StringArray>
['음식', '시설관리·임대', '예술·스포츠', '소매', '부동산', '과학·기술', '수리·개인', '보건의료', '숙박', '교육']
Length: 10, dtype: str

### 로컬DB(mysql) 기본 csv 파일 넣기

In [5]:
# 로컬 DB와 연동
# 로컬 DB 정보 입력 
user = 'root'     
password = 'mysqlmysql'  
host = '127.0.0.1'
port = '3306'
database = 'ETL_Project'

# DB 연결
localengine = create_engine(f'mysql+pymysql://{user}:{password}@{host}:{port}/{database}')

# 연결 테스트
conn = localengine.connect()
print("로컬 DB 연결 성공!")
conn.close()

로컬 DB 연결 성공!


In [7]:
#csv 파일 저장
tableName = "raw_store_data"
df.to_sql (name = tableName, con=localengine, if_exists = "replace" , index=False)

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

### 데이터 정재

In [6]:
#'화곡동, 음식' 데이터만 필터링에서 확인
hwagok_pood_df = df[(df['상권업종대분류명'] == '음식') & (df['법정동명'] == '화곡동')]

#데이터 갯수 확인
print(f"화곡동 음식점: {len(hwagok_pood_df)}개")

화곡동 음식점: 2364개


In [8]:
#상호명,상권업종대분류명,상권업종중분류코드,상권업종소분류명,지번주소 컬럼만 추출한 데이터 생성
target_cols = ['상호명','상권업종대분류명','상권업종중분류명','상권업종소분류명','지번주소']
refined_df = df[(df['상권업종대분류명'] == '음식') & (df['법정동명'] == '화곡동')][target_cols].copy()

In [9]:
#데이터 갯수 비교 검증
print(f"화곡동 음식점:{len(refined_df)} 개")

화곡동 음식점:2364 개


### 클라우드 DB(mysql) 정재 csv 파일 넣기

In [10]:
# 클라우드 DB 정보 입력 
user = 'kopouser'     
password = 'kopouser'  
host = '3.26.13.129'
port = '3306'
database = 'ETL_Project'

# DB 연결
cloudengine = create_engine(f'mysql+pymysql://{user}:{password}@{host}:{port}/{database}')

# 연결 테스트
conn = localengine.connect()
print("클라우드 DB 연결 성공!")
conn.close()

클라우드 DB 연결 성공!


In [ ]:
#csv 파일 저장
tableName = "refined_stores"
#refined_df.to_sql (name = tableName, con=cloudengine, if_exists = "replace" , index=False)

NameError: name 'refined_df' is not defined

### fast API(백앤드) 서버 구축

In [11]:
app = FastAPI() 

# CORS 미들웨어 추가
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # 모든 origin 허용
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["*"],
) 

In [ ]:
#fast Api 연결 확인용 get
@app.get("/")
def read_root():
    return {"message": "ETL API Server is Running!"}

#실제 구동 api 
@app.get("/search")
def search_stores(category: str): 

    query =f"SELECT * FROM refined_stores where `상권업종중분류명` = '{category}'"
    data = pd.read_sql(query, cloudengine)

    #JSON 저장
    return data.to_dict(orient="records")

nest_asyncio.apply()

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8002)

INFO:     Started server process [18668]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)
